In [ ]:
# ============================================================
# Multimodal-style Vasopressor Prediction Model (MDS-ED)
# Using tabular routine clinical data only
#
# Target:
#   deterioration_vasopressors
#
# Features:
#   demographics + biometrics + vitals + lab values
#
# Model:
#   RealMLP-style dense neural network
#
# Metrics:
#   AUROC
#   AUPRC
#   Brier Score
#   Uncertainty (entropy)
#
# Memory-safe version
# ============================================================

import pandas as pd
import numpy as np
import warnings

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

# ============================================================
# 1. Load data
# ============================================================

print("Loading dataset...")

df = pd.read_csv(
    "/content/mds_ed.csv",
    low_memory=False
)

print("Dataset shape:", df.shape)

# ============================================================
# 2. Define feature groups
# ============================================================

demographics_columns = [
    c for c in df.columns if "demographics_" in c
]

biometrics_columns = [
    c for c in df.columns if "biometrics_" in c
]

vitals_columns = [
    c for c in df.columns if "vitals_" in c
]

labvalues_columns = [
    c for c in df.columns if "labvalues_" in c
]

# Combine all selected features
features = (
    demographics_columns +
    biometrics_columns +
    vitals_columns +
    labvalues_columns
)

# Target column
target_col = "deterioration_vasopressors"

print("Number of features:", len(features))
print("Target:", target_col)

# ============================================================
# 3. Keep valid labels only
# ============================================================

df = df[df[target_col] != -999]

# Remove NaN labels
df = df[df[target_col].notna()]

# Convert target to integer
df[target_col] = df[target_col].astype(int)

print("Remaining rows:", len(df))

# ============================================================
# 4. Train / Validation / Test split
# ============================================================

train_df = df[
    df["general_strat_fold"].isin(range(0, 18))
].reset_index(drop=True)

val_df = df[
    df["general_strat_fold"] == 18
].reset_index(drop=True)

test_df = df[
    df["general_strat_fold"] == 19
].reset_index(drop=True)

# ============================================================
# 5. Use first ECG only for validation/test
# ============================================================

val_df = val_df[
    val_df["general_ecg_no_within_stay"] == 0
]

test_df = test_df[
    test_df["general_ecg_no_within_stay"] == 0
]

# ============================================================
# 6. Extract arrays
# ============================================================

x_train = train_df[features]
x_val = val_df[features]
x_test = test_df[features]

y_train = train_df[target_col].values
y_val = val_df[target_col].values
y_test = test_df[target_col].values

print("Train:", x_train.shape)
print("Validation:", x_val.shape)
print("Test:", x_test.shape)

# ============================================================
# 7. Median imputation
# ============================================================

imputer = SimpleImputer(strategy="median")

x_train = imputer.fit_transform(x_train)
x_val = imputer.transform(x_val)
x_test = imputer.transform(x_test)

# ============================================================
# 8. Feature normalization
# ============================================================

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

# ============================================================
# 9. Convert to PyTorch tensors
# ============================================================

x_train = torch.tensor(x_train, dtype=torch.float32)
x_val = torch.tensor(x_val, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# ============================================================
# 10. Create dataloaders
# ============================================================

batch_size = 256

train_loader = DataLoader(
    TensorDataset(x_train, y_train),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(x_val, y_val),
    batch_size=batch_size,
    shuffle=False
)

# ============================================================
# 11. RealMLP-style model
# ============================================================

class RealMLP(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),

            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.network(x)

model = RealMLP(input_dim=x_train.shape[1])

# ============================================================
# 12. Loss and optimizer
# ============================================================

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# ============================================================
# 13. Training loop
# ============================================================

epochs = 20

print("\nTraining model...\n")

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for xb, yb in train_loader:

        optimizer.zero_grad()

        outputs = model(xb).squeeze()

        loss = criterion(outputs, yb)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss: {total_loss:.4f}"
    )

# ============================================================
# 14. Prediction
# ============================================================

model.eval()

with torch.no_grad():

    logits = model(x_test).squeeze()

    y_pred = torch.sigmoid(logits).numpy()

# ============================================================
# 15. Metrics
# ============================================================

auroc = roc_auc_score(y_test, y_pred)

auprc = average_precision_score(y_test, y_pred)

brier = brier_score_loss(y_test, y_pred)

# ============================================================
# 16. Uncertainty estimation
#
# Entropy:
#   High entropy  -> uncertain prediction
#   Low entropy   -> confident prediction
# ============================================================

epsilon = 1e-10

entropy = -(
    y_pred * np.log(y_pred + epsilon)
    +
    (1 - y_pred) * np.log(1 - y_pred + epsilon)
)

mean_uncertainty = entropy.mean()

# ============================================================
# 17. Results
# ============================================================

results = pd.DataFrame([{
    "target": target_col,
    "auroc": auroc,
    "auprc": auprc,
    "brier_score": brier,
    "mean_uncertainty": mean_uncertainty,
    "test_n": len(y_test),
    "positive_rate": y_test.mean()
}])

results.to_csv(
    "vasopressor_multimodal_results.csv",
    index=False
)

print("\n===== FINAL RESULTS =====")

print(results.to_string(index=False))

print("\nSaved to:")
print("vasopressor_multimodal_results.csv")